# Research Notebook for PISA question and answers on different languages

## Libraries

In [14]:
import os, getpass

In [15]:
from dotenv import load_dotenv, find_dotenv

In [7]:
from openai import OpenAI

In [ ]:
load_dotenv(find_dotenv(usecwd=True))  # finds .env from current working dir upward

In [16]:
# --- 0) Setup: imports & config
import os, re, time, json, math, random
from typing import Dict, List, Tuple, Any, Optional
import pandas as pd
import time

In [17]:
from tqdm import tqdm

## Configuration

In [12]:
client = OpenAI() 

In [34]:
MODEL_GPT = "gpt-5.1-2025-11-13"                    # <- replace with exact model id if different

In [103]:
SHEET_ID = "1QVPzB7uMwqJ6jCsHkwIILnXvDQIycpqkcV3bkiDpzyQ"
WORKSHEET_NAME = "dataset"  # change if needed
RESULTS_CSV = "llm_eval_results_GPT5.csv"
SAMPLE_PER_LANGUAGE = 2     # 5 per language
MAX_LANGUAGES = 4          # 43 languages total
SEED = 42

random.seed(SEED)

## Load data from Google Sheet

In [69]:
csv_url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv&sheet={WORKSHEET_NAME}"
try:
    df = pd.read_csv(csv_url)
except Exception as e:
    raise RuntimeError(
        "Failed to read the Google Sheet via CSV export. "
        "Make sure the sheet is shared as 'Anyone with the link can view', "
        f"ID is correct, and tab name matches. Underlying error: {e}"
    )

expected_cols = {
    "qid","language","question","context","options","gold",
    "answer_type","category","difficulty","rationale","source"
}
missing = expected_cols - set(df.columns)
if missing:
    raise ValueError(f"Your sheet is missing columns: {sorted(missing)}")

In [70]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1075 entries, 0 to 1074
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   qid            1075 non-null   object
 1   language       1075 non-null   object
 2   language_code  1075 non-null   object
 3   question       1075 non-null   object
 4   context        1075 non-null   object
 5   options        1075 non-null   object
 6   gold           1075 non-null   object
 7   answer_type    1075 non-null   object
 8   category       1075 non-null   object
 9   difficulty     1075 non-null   object
 10  rationale      172 non-null    object
 11  source         1075 non-null   object
dtypes: object(12)
memory usage: 100.9+ KB


## Normalize & sample

In [104]:
df["language"] = df["language"].astype(str).str.strip()
# pick first MAX_LANGUAGES languages (sorted)
languages = sorted(df["language"].unique())[:MAX_LANGUAGES]

In [105]:
# sample up to SAMPLE_PER_LANGUAGE per language
sampled = (
    df[df["language"].isin(languages)]
    .groupby("language", sort=True, group_keys=False)
    .head(SAMPLE_PER_LANGUAGE)
    .reset_index(drop=True)
)
if sampled.empty:
    raise ValueError("No rows selected. Check your data.")

In [106]:
print(f"Selected {len(sampled)} rows across {sampled['language'].nunique()} languages.")
display(sampled[["qid","language","question","gold"]])

Selected 8 rows across 4 languages.


,qid,language,question,gold
0,q001,Albanian,Nëse Tania vendos të blejë makinën D dhe ta sh...,C
1,q002,Albanian,"Nëse trendi i shitjeve vazhdon, në cilin vit n...",C
2,q001,Arabic,إذا قررت تهاني شراء السيارة D وإعادة بيعها بع...,C
3,q002,Arabic,في حالة استمرار المبيعات في هذا الاتجاه، في أي...,C
4,q001,Azerbaijani / Azeri,Əgər Turan maşın D-ni almağa və üç il sonra əl...,C
5,q002,Azerbaijani / Azeri,"Əgər bu satış tendensiyası davam edərsə, yuxar...",C
6,q001,Basque,Taniak erabakitzen badu D autoa erostea eta ho...,C
7,q002,Basque,"Salmenten joera honek berdin jarraitzen badu, ...",C


## Parse options

In [50]:
def parse_options(raw: str) -> Dict[str, Any]:
    """
    Parse multiple-choice options from a JSON-encoded string.

    Supported formats
    -----------------
    1) Simple labeled strings (original format):
        ["A) 1575", "B) 8925", "C) 9000", "D) 9975"]

        -> {"A": "1575", "B": "8925", "C": "9000", "D": "9975"}

    2) List of dicts with labels mapping to lists of tokens:
        [
          {"A": ["India", "Colombia"]},
          {"B": ["India", "Armenia"]},
          {"C": ["Panama", "Colombia"]},
          {"D": ["Kazakhstan", "Colombia"]}
        ]
    """
    if pd.isna(raw):
        raise ValueError("Options are empty")

    # Load JSON
    try:
        items = json.loads(raw)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON format for options: {e}")

    if not isinstance(items, list):
        raise ValueError("Expected a JSON list at top level")

    # Case 1: list of strings -> original behavior
    if all(isinstance(item, str) for item in items):
        options: Dict[str, Any] = {}
        for item in items:
            # Match patterns like "A) text", "B. text", or "C: text"
            if not isinstance(item, str):
                raise ValueError(f"Option is not a string: {item}")
            match = re.match(r"^\s*([A-Z])[\)\.\:]\s*(.+)$", item.strip())
            if match:
                label, text = match.groups()
                options[label.upper()] = text.strip()
            else:
                # Fallback: assign next available letter automatically
                next_label = chr(ord('A') + len(options))
                options[next_label] = item.strip()
        return options

    # Case 2: list of dicts like [{"A": [...]}, {"B": [...]}]
    if all(isinstance(item, dict) for item in items):
        options: Dict[str, Any] = {}
        for idx, d in enumerate(items):
            if len(d) != 1:
                raise ValueError(
                    f"Each dict must have exactly one key (label). Problem at index {idx}: {d}"
                )
            (label_raw, value) = next(iter(d.items()))
            if not isinstance(label_raw, str):
                raise ValueError(f"Label must be a string, got: {label_raw}")

            label = label_raw.strip().upper()
            if not re.fullmatch(r"[A-Z]", label):
                raise ValueError(f"Invalid option label '{label_raw}' at index {idx}")

            # Accept list or scalar; normalize scalars into single-element lists if needed
            if isinstance(value, list):
                options[label] = value
            else:
                options[label] = [value]

        return options

    # If we reach here, the list is mixed or has unsupported types
    raise ValueError(
        "Unsupported options format: expected list of strings or list of single-key dicts"
    )


In [28]:
test = parse_options('["A) 2018", "B) 2019", "C) 2020", "D) 2021"]')
print(test)

{'A': '2018', 'B': '2019', 'C': '2020', 'D': '2021'}


In [29]:
options_block = "\n".join([f"{k}. {v}" for k,v in test.items()])
print(options_block)

A. 2018
B. 2019
C. 2020
D. 2021


## Build prompt

In [51]:
def build_prompt(row: pd.Series, options: Dict[str,str]) -> str:
    """
    Builds prompt for MCQ.
    """
    options_block = "\n".join([f"{k}. {v}" for k,v in options.items()])

    return (
        f"{row['context']}\n"
        f"{row['question']}\n"
        f"{options_block}\n\n"
    )

In [52]:
answer_letter_regex = re.compile(r"<\s*([A-Z])\s*[.)]?\s*>")

def extract_letter(text: str, valid_letters: List[str]) -> str:
    """
    Extract the first single-letter A-Z token that is in valid_letters.
    """
    if not text:
        return ""
    # First line is the letter per our format; but still be defensive:
    first_line = text.splitlines()[0].strip().rstrip(".)").upper()
    # If first line is a single valid letter, use it
    if len(first_line) == 1 and first_line.upper() in valid_letters:
        return first_line.upper()
    # Else find any A-Z token
    m = answer_letter_regex.search(text.upper())
    if m and m.group(1) in valid_letters:
        return m.group(1)
    return ""

## LLM call wrapper

In [ ]:
def llm_completion(
    prompt: str,
    model: Optional[str] = None,
    max_tokens: Optional[int] = None,
    system_prompt: str="Reply format: <LETTER>", 
    retries: int = 2,
    backoff_seconds: float = 1.5,
    **kwargs,
) -> str:
    """
    Call an LLM via chat.completions and return text.
    - `prompt`: user content (string)
    - `model`: overrides global MODEL_NAME if provided
    - `system_prompt`: system role content
    - `retries`: retry on transient errors
    - `backoff_seconds`: base backoff between retries
    - `**kwargs`: forwarded to client.chat.completions.create (e.g., stop, seed)
    """
    mdl = model or MODEL_GPT
    last_err = None

    for attempt in range(retries + 1):
        try:
            # Build arguments dynamically — include only if provided
            call_args = dict(
                model=mdl,
                input=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": prompt},
                ],
                stream=False
            )

            # ---- attach token limits if provided ----
            if max_tokens is not None:
                call_args["max_tokens"] = max_tokens

            # Merge other kwargs (e.g., stop, seed, etc.)
            for k, v in kwargs.items():
                if k not in call_args:
                    call_args[k] = v

            # Actual model call
            response = client.responses.create(**call_args)
            if response is None:
                return "No response from model."
            else:
                return(response.output_text or "").strip()

        except Exception as e:
            last_err = e
            if attempt < retries:
                time.sleep(backoff_seconds * (attempt + 1))
            else:
                raise

## Evaluation loop

In [54]:
def eval_rows(
        rows: pd.DataFrame, 
        cycle: int,
        model_tag: str,
        model_name: str, 
        results_path: str = RESULTS_CSV,
        sleep_s: float = 0.0, 
        retries: int = 2
    ) -> pd.DataFrame:
    results = []
    file_exists = os.path.exists(results_path)
    for i, row in tqdm(rows.iterrows(), total=len(rows), desc=f"Evaluating {model_tag} / {model_name}, cycle {cycle}"):
        qid = row["qid"]
        lang = row["language"]
        gold = str(row["gold"]).strip().upper()

        # Parse options
        try:
            opts = parse_options(row["options"])
        except Exception as e:
            row_result = {
                "qid": qid,
                "language": lang,
                "pred": "",
                "gold": gold,
                "is_correct": False,
                "error": f"OptionsParseError: {e}",
                "raw": "",
                "question": row["question"],
                "options_json": json.dumps(opts if 'opts' in locals() else {}, ensure_ascii=False),
                "model_tag": model_tag,
                "model_name": model_name,
                "cycle": cycle,
            }
            results.append(row_result)

            # Save immediately
            pd.DataFrame([row_result]).to_csv(
                results_path,
                mode="a",
                header=not file_exists,
                index=False,
                encoding="utf-8"
            )
            file_exists = True
            continue

        valid_letters = sorted(list(opts.keys()))
        prompt = build_prompt(row, opts)

        # call model with simple retry
        raw = ""
        err = ""
        for attempt in range(retries + 1):
            try:
                raw = llm_completion(prompt, model = model_name)
                break
            except Exception as e:
                err = f"{type(e).__name__}: {e}"
                if attempt < retries:
                    time.sleep(1.5 * (attempt + 1))
                else:
                    raw = ""
        pred = extract_letter(raw, valid_letters)
        is_correct = (pred == gold)

        row_result = {
            "qid": qid,
            "language": lang,
            "pred": pred,
            "gold": gold,
            "is_correct": bool(is_correct),
            "error": err,
            "raw": raw,
            "question": row["question"],
            "options_json": json.dumps(opts, ensure_ascii=False),
            "model_tag": model_tag,
            "model_name": model_name,
            "cycle": cycle,
        }

        results.append(row_result)

        # *** Save this row immediately ***
        pd.DataFrame([row_result]).to_csv(
            results_path,
            mode="a",
            header=not file_exists,
            index=False,
            encoding="utf-8"
        )
        file_exists = True

        if sleep_s > 0:
            time.sleep(sleep_s)
    return pd.DataFrame(results)

## Execution and results

In [55]:
MODELS_TO_TEST = [
    ("GPT", MODEL_GPT)
]

In [ ]:
# filtered_df = df[df["language"] == "English"]
# sampled = df

In [56]:
N_CYCLES = 1  # repeat the same question to the same LLM

In [114]:
if os.path.exists(RESULTS_CSV):
    existing_results = pd.read_csv(RESULTS_CSV)
    print(f"Loaded existing results from {RESULTS_CSV}: {len(existing_results)} rows")
else:
    existing_results = pd.DataFrame()
    print("No existing results file found. Starting fresh.")

for tag, model_name in MODELS_TO_TEST:
    print(f"\nEvaluating {tag} -> {model_name}")
    for cycle in range(1, N_CYCLES + 1):
        # Determine which qids are already done for this (tag, model_name, cycle)
        if existing_results.empty:
            # Nothing done yet at all
            rows_to_eval = sampled.copy()
        else:
            # Subset only rows already done for this (tag, model_name, cycle)
            subset = existing_results[
                (existing_results["model_tag"] == tag) &
                (existing_results["model_name"] == model_name) &
                (existing_results["cycle"] == cycle)
            ]

            if subset.empty:
                # No rows done yet for this model+cycle
                rows_to_eval = sampled.copy()
            else:
                # Use (qid, language) pairs as the key — more robust than qid alone
                done_pairs = set(zip(subset["qid"], subset["language"]))

                mask = ~sampled.apply(
                    lambda r: (r["qid"], r["language"]) in done_pairs,
                    axis=1
                )
                rows_to_eval = sampled[mask]

        if rows_to_eval.empty:
            print(f"  • cycle {cycle}/{N_CYCLES}: already complete, skipping")
            continue

        print(f"  • cycle {cycle}/{N_CYCLES}: evaluating {len(rows_to_eval)} questions")
        t0 = time.time()

        df_new = eval_rows(
            rows_to_eval,
            model_tag=tag,
            model_name=model_name,
            cycle=cycle,
            results_path=RESULTS_CSV
        )

        elapsed = time.time() - t0
        print(f"    Done in {elapsed:.1f}s, newly evaluated {len(df_new)} rows.")

        # Update in-memory copy so subsequent cycles/ models can see freshly written rows
        existing_results = pd.concat([existing_results, df_new], ignore_index=True)

No existing results file found. Starting fresh.

Evaluating GPT -> gpt-5.1-2025-11-13
  • cycle 1/1: evaluating 8 questions


Evaluating GPT / gpt-5.1-2025-11-13, cycle 1: 100%|██████████| 8/8 [00:12<00:00,  1.50s/it]

    Done in 12.0s, newly evaluated 8 rows.


In [108]:
res_df = pd.read_csv(RESULTS_CSV)
print(f"\nTotal results loaded: {len(res_df)}")


Total results loaded: 8


In [110]:
overall_by_model = (
    res_df.groupby(["model_tag","model_name"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values("accuracy", ascending=False)
)
print("\nOverall accuracy by model:")
display(overall_by_model)


Overall accuracy by model:


,model_tag,model_name,accuracy
0,GPT,gpt-5.1-2025-11-13,0.0


In [111]:
overall_by_question = (
    res_df.groupby(["qid"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values("accuracy", ascending=False)
)
print("\nOverall accuracy by question:")
display(overall_by_question)


Overall accuracy by question:


,qid,accuracy
0,q001,0.0
1,q002,0.0


In [101]:
overall_by_lang = (
    res_df.groupby(["language"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values("accuracy", ascending=False)
)
print("\nOverall accuracy by lang:")
display(overall_by_lang)


Overall accuracy by lang:


,language,accuracy
0,Albanian,0.0
1,Arabic,0.0
2,Azerbaijani / Azeri,0.0
3,Basque,0.0
4,Bokmål,0.0
5,Bosnian,0.0
6,Bulgarian,0.0
7,Catalan,0.0
8,Chinese,0.0
9,Croatian,0.0


In [42]:
by_model_lang = (
    res_df.groupby(["model_tag","model_name","language"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values(["model_tag","language"])
)
print("\nAccuracy by model & language:")
display(by_model_lang)


Accuracy by model & language:


,model_tag,model_name,language,accuracy
0,GPT,gpt-5.1-2025-11-13,Albanian,0.0
1,GPT,gpt-5.1-2025-11-13,Arabic,0.0
2,GPT,gpt-5.1-2025-11-13,Azerbaijani / Azeri,0.0
3,GPT,gpt-5.1-2025-11-13,Basque,0.0
4,GPT,gpt-5.1-2025-11-13,Bokmål,0.0


In [43]:
by_question = (
    res_df.groupby(["model_tag","model_name","language","qid"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values(["accuracy"])
)

# print("\nAccuracy by question:")
# display(by_question)

accuracy_counts_total = (
    by_question["accuracy"]
    .value_counts()
    .rename_axis("accuracy")
    .reset_index(name="count")
    .sort_values("accuracy")
)

print("\nCount of total questions by accuracy:")
display(accuracy_counts_total)

accuracy_counts = (
    by_question
    .groupby(["model_tag", "model_name", "accuracy"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .sort_values(["model_tag", "model_name", "accuracy"])
)

print("\nCount of questions by accuracy:")
display(accuracy_counts)


Count of total questions by accuracy:


,accuracy,count
0,0.0,5



Count of questions by accuracy:


,model_tag,model_name,accuracy,count
0,GPT,gpt-5.1-2025-11-13,0.0,5


In [44]:
def safe_acc(s):
    return float('nan') if s.empty else s.mean()
overall_acc = safe_acc(res_df["is_correct"])
print(f"\nCombined overall accuracy on {len(res_df)} items: {overall_acc:.3f}")


Combined overall accuracy on 5 items: 0.000


In [45]:
for tag, _ in MODELS_TO_TEST:
    out_path = f"llm_eval_results__{tag}.csv"
    res_df.query("model_tag == @tag").to_csv(out_path, index=False)
    print(f"Saved {tag} results to: {out_path}")

# (Optional) quick peek
display(res_df.head())

Saved GPT results to: llm_eval_results__GPT.csv


,qid,language,pred,gold,is_correct,error,raw,question,options_json,model_tag,model_name,cycle
0,q001,Albanian,B,C,False,NaN,B,Nëse Tania vendos të blejë makinën D dhe ta sh...,"{""A"": ""1575"", ""B"": ""8925"", ""C"": ""9000"", ""D"": ""...",GPT,gpt-5.1-2025-11-13,1
1,q001,Arabic,B,C,False,NaN,B,إذا قررت تهاني شراء السيارة D وإعادة بيعها بع...,"{""A"": ""1575"", ""B"": ""8925"", ""C"": ""9000"", ""D"": ""...",GPT,gpt-5.1-2025-11-13,1
2,q001,Azerbaijani / Azeri,B,C,False,NaN,B,Əgər Turan maşın D-ni almağa və üç il sonra əl...,"{""A"": ""1575"", ""B"": ""8925"", ""C"": ""9000"", ""D"": ""...",GPT,gpt-5.1-2025-11-13,1
3,q001,Basque,B,C,False,NaN,B,Taniak erabakitzen badu D autoa erostea eta ho...,"{""A"": ""1575"", ""B"": ""8925"", ""C"": ""9000"", ""D"": ""...",GPT,gpt-5.1-2025-11-13,1
4,q001,Bokmål,B,C,False,NaN,B,Omtrent hvor mye vil bruktprisen av bilen være...,"{""A"": ""1575"", ""B"": ""8925"", ""C"": ""9000"", ""D"": ""...",GPT,gpt-5.1-2025-11-13,1


In [93]:
df_false = res_df[res_df['is_correct'] == False]
display(df_false.head(50))

,qid,language,pred,gold,is_correct,error,raw,question,options_json,model_tag,model_name,cycle
10,q011,English,A,B,False,NaN,A,Helena claims that South Korea has more forest...,"{""A"": ""Yes"", ""B"": ""No""}",GPT,gpt-5,1
35,q011,Albanian,A,B,False,NaN,A,Helena mendon se Koreja e Jugut ka më tepër si...,"{""A"": ""Po"", ""B"": ""Jo""}",GPT,gpt-5,1
46,q022,Albanian,D,C,False,NaN,D,Pse vendos Ivana_88 të postojë pyetjen e saj n...,"{""A"": ""Sepse nuk di si të gjejë një veteriner....",GPT,gpt-5,1
59,q010,Arabic,C,A,False,NaN,C,بالنظر في الفترتين الزمنيتين: 2005 إلى 2010...,"{""A"": [""الهند"", ""كولومبيا ""], ""B"": [""الهند"", ""...",GPT,gpt-5,1
60,q011,Arabic,A,B,False,NaN,<A>,تدّعي حليمة أنّ كوريا الجنوبية لديها مساحة سطح...,"{""A"": ""نعم"", ""B"": ""لا""}",GPT,gpt-5,1
85,q011,Azerbaijani / Azeri,A,B,False,NaN,A,"Həlimə iddia edir ki, göstərilmiş illər üçün d...","{""A"": ""Bəli"", ""B"": ""Xeyr""}",GPT,gpt-5,1
97,q023,Azerbaijani / Azeri,C,A,False,NaN,C,Məlumat İlahə_88-in probleminə uyğundurmu?\nNa...,"{""A"": [""Bəli"", ""Bəli"", ""Xeyr"", ""Xeyr"", ""Bəli""]...",GPT,gpt-5,1
110,q011,Basque,A,B,False,NaN,A,Haizeak baieztatu du Hego Koreak zerrendako be...,"{""A"": ""Bai"", ""B"": ""Ez""}",GPT,gpt-5,1
128,q004,Bokmål,D,C,False,NaN,D,"Firmaet som leier ut flyttebilene, har bekreft...","{""A"": ""Hun har rett, fordi høyden til en eske ...",GPT,gpt-5,1
160,q011,Bosnian,A,B,False,NaN,A,Helena tvrdi da Južna Koreja ima više površine...,"{""A"": ""Da"", ""B"": ""Ne""}",GPT,gpt-5,1


In [96]:
df_summary = (
    df_false
    .groupby(['qid'])
    .size()
    .reset_index(name='num_incorrect')
)

print(df_summary)


    qid  num_incorrect
0  q001              1
1  q004              2
2  q010              7
3  q011             29
4  q014              1
5  q018              3
6  q019              2
7  q022              3
8  q023              2
